# 08 — Download manually identified Commission decisions

This notebook is the **raw-data ingestion step** for the manually identified European Commission decisions listed in:

`input/commission_manual_injection.csv`

Its job is deliberately narrow:

1. read the manually curated input table;
2. expand one or more source URLs per decision;
3. download the source documents into `data/raw/com_manual/files/`;
4. validate that the returned content looks like a real document rather than a robot-check or placeholder page;
5. write a compact download manifest to `output/com_manual/com_manual_download_manifest.csv`.

The notebook does **not** extract text, run OCR, clean documents, or create analytical variables. Those tasks belong to the subsequent `process_com_manual` step.

### Why keep this step separate?

For reproducible scientific data construction, it is useful to distinguish **source acquisition** from **data processing**. The raw files saved here are the source material from which the later cleaned text dataset is produced.

EUR-Lex can occasionally return JavaScript, verification, or document-generation pages instead of the requested legal document. For that reason, EUR-Lex URLs are opened through Playwright/Chromium and the returned content is validated before it is accepted.

> **Dependency:** if Playwright is not installed, run `pip install playwright` followed by `playwright install chromium` once in your environment.

In [1]:
from pathlib import Path
import asyncio
import hashlib
import random
import re
from urllib.parse import parse_qs, parse_qsl, unquote, urlencode, urlparse, urlunparse

import pandas as pd
import requests
from tqdm.auto import tqdm

try:
    from playwright.async_api import async_playwright
except Exception:
    async_playwright = None
    print(
        "WARNING: Playwright is not available. "
        "Install it with: pip install playwright && playwright install chromium"
    )

## 1. Settings and project paths

The notebook expects to run somewhere inside the `eccjeu` repository. It finds the project root by looking for `input/commission_manual_injection.csv`.

Only the paths and a few download settings should normally need adjustment.

In [2]:
# -----------------------------
# User settings
# -----------------------------

PROJECT_ROOT = None  # e.g. Path("/home/edik/projects/eccjeu") if auto-detection fails

INPUT_REL_PATH = Path("input/commission_manual_injection.csv")
RAW_REL_DIR = Path("data/raw/com_manual/files")
MANIFEST_REL_PATH = Path("output/com_manual/com_manual_download_manifest.csv")

REDO_ALL = False

# Be polite to the source server.
SLEEP_SECONDS = (1.5, 4.0)

# EUR-Lex browser settings.
HEADLESS = True
PLAYWRIGHT_TIMEOUT_MS = 90_000
PLAYWRIGHT_RENDER_WAIT_SECONDS = 7.5
PLAYWRIGHT_EXTRA_SETTLE_SECONDS = 2.0
PLAYWRIGHT_MAX_RENDER_RETRIES = 3
PLAYWRIGHT_RETRY_SLEEP_SECONDS = 10.0

# Basic document validation.
MIN_SUCCESS_BYTES = 1
MIN_HTML_TEXT_CHARS = 500

# Direct-download settings for non-EUR-Lex URLs.
REQUEST_TIMEOUT_SECONDS = 45

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/126.0 Safari/537.36"
)

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,"
              "application/pdf;q=0.8,*/*;q=0.7",
    "Accept-Language": "en-US,en;q=0.9,de;q=0.8,fr;q=0.7",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}


def find_project_root(start: Path | None = None) -> Path:
    if PROJECT_ROOT is not None:
        return Path(PROJECT_ROOT).expanduser().resolve()

    starts = [Path.cwd().resolve()]
    if start is not None:
        starts.insert(0, Path(start).resolve())

    for base in starts:
        for candidate in [base, *base.parents]:
            if (candidate / INPUT_REL_PATH).exists():
                return candidate

    fallback = Path("/home/edik/projects/eccjeu")
    if (fallback / INPUT_REL_PATH).exists():
        return fallback

    raise FileNotFoundError(
        f"Could not find {INPUT_REL_PATH}. "
        "Set PROJECT_ROOT manually in the settings cell."
    )


project_root = find_project_root()
input_path = project_root / INPUT_REL_PATH
raw_dir = project_root / RAW_REL_DIR
manifest_path = project_root / MANIFEST_REL_PATH

raw_dir.mkdir(parents=True, exist_ok=True)
manifest_path.parent.mkdir(parents=True, exist_ok=True)

print(f"Project root : {project_root}")
print(f"Input CSV    : {input_path}")
print(f"Raw files    : {raw_dir}")
print(f"Manifest     : {manifest_path}")

Project root : /home/edik/projects/ec_cjeu
Input CSV    : /home/edik/projects/ec_cjeu/input/commission_manual_injection.csv
Raw files    : /home/edik/projects/ec_cjeu/data/raw/com_manual/files
Manifest     : /home/edik/projects/ec_cjeu/output/com_manual/com_manual_download_manifest.csv


## 2. Load the manual Commission list

The input table is treated as a curated source list. Its descriptive fields are carried forward unchanged into the download manifest.

The expected columns are:

- `decision_date`
- `case_numbers_raw`
- `case_numbers_normalized`
- `decision_label`
- `decision_title`
- `download_urls`

A row may contain more than one URL. Each URL becomes one download record. If a row has no URL, it is still retained in the manifest as an unsuccessful `missing_url` record.

In [3]:
manual_df = pd.read_csv(input_path)
manual_df.columns = [str(c).strip() for c in manual_df.columns]

required_cols = {
    "decision_date",
    "case_numbers_raw",
    "case_numbers_normalized",
    "decision_label",
    "decision_title",
    "download_urls",
}

missing = required_cols - set(manual_df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

print(f"Input rows: {len(manual_df):,}")
manual_df.head()

Input rows: 16


,decision_date,case_numbers_raw,case_numbers_normalized,decision_label,decision_title,download_urls
0,11.12.1981,IV/511,511.0,Infringement,National Panasonic France,https://eur-lex.europa.eu/legal-content/EN/TXT...
1,14.12.1999,IV/34.237,34237.0,Infringement,Anheuser-Busch Inc/Scottish and New Castle,https://eur-lex.europa.eu/legal-content/EN/TXT...
2,28.06.1995,NaN,NaN,Infringement,Zaventem,https://eur-lex.europa.eu/legal-content/EN/TXT...
3,24.04.1985,NaN,NaN,Infringement,Greek Insurance,https://eur-lex.europa.eu/legal-content/EN/TXT...
4,22.06.1987,NaN,NaN,Infringement,Canary Islands,https://eur-lex.europa.eu/legal-content/EN/TXT...


## 3. Prepare the download registry

Filenames are kept simple and source-oriented.

- If the URL already contains a useful filename, that filename is retained.
- For EUR-Lex document endpoints without a useful basename, the CELEX identifier is used, for example `31982D0465.html`.
- If two input records would write to the same filename, a readable row/URL suffix is added so that no source record is silently overwritten.

The actual download happens in the next section. This step only prepares the records and lets us inspect what is about to happen.

In [4]:
def clean_value(value) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return str(value).strip()


def split_urls(value) -> list[str]:
    if pd.isna(value):
        return []

    text = str(value).strip()
    if not text:
        return []

    parts = re.split(r"\s*[|\n\r]+\s*", text)
    urls = []

    for part in parts:
        part = part.strip()
        if not part:
            continue
        urls.extend(
            bit.strip()
            for bit in re.split(r"\s+(?=https?://)", part)
            if bit.strip()
        )

    return urls


def extract_celex(url: str) -> str | None:
    decoded = unquote(clean_value(url))

    patterns = [
        r"CELEX[:=]([0-9A-Z().\-]+)",
        r"uri=CELEX:?([0-9A-Z().\-]+)",
        r"celex=([0-9A-Z().\-]+)",
    ]

    for pattern in patterns:
        match = re.search(pattern, decoded, flags=re.IGNORECASE)
        if match:
            return match.group(1).upper().strip().rstrip("&/?#")

    parsed = urlparse(decoded)
    query = parse_qs(parsed.query)

    for key in ("uri", "celex"):
        for value in query.get(key, []):
            value = unquote(value)
            match = re.search(
                r"CELEX:?([0-9A-Z().\-]+)",
                value,
                flags=re.IGNORECASE,
            )
            if match:
                return match.group(1).upper().strip().rstrip("&/?#")

            if re.match(r"^[0-9][0-9A-Z().\-]+$", value, flags=re.IGNORECASE):
                return value.upper().strip().rstrip("&/?#")

    return None


def safe_filename(value: str, fallback: str = "download") -> str:
    text = unquote(clean_value(value))
    text = text.split("?")[0].split("#")[0]
    text = re.sub(r'[\\/:*?"<>|]+', "_", text)
    text = re.sub(r"\s+", "_", text)
    text = text.strip(" ._")
    return text or fallback


def extension_from_url(url: str) -> str:
    decoded = unquote(clean_value(url))
    path = urlparse(decoded).path.lower().rstrip("/")
    suffix = Path(path).suffix.lower()

    if suffix in {".pdf", ".html", ".htm", ".xml", ".txt", ".doc", ".docx"}:
        return suffix

    if "/txt/html" in decoded.lower() or path.endswith("/html"):
        return ".html"

    if "/txt/pdf" in decoded.lower() or path.endswith("/pdf"):
        return ".pdf"

    return ".bin"


def source_filename_from_url(url: str, source_row: int) -> str:
    decoded = unquote(clean_value(url))

    if not decoded:
        return f"missing_url_row_{source_row}.bin"

    parsed = urlparse(decoded)
    basename = Path(parsed.path.rstrip("/")).name
    extension = extension_from_url(decoded)

    generic_endpoint_names = {"HTML", "PDF", "TXT", "ALL"}

    if (
        basename
        and basename.upper() not in generic_endpoint_names
        and "." in basename
    ):
        return safe_filename(basename)

    celex = extract_celex(decoded)
    if celex:
        return safe_filename(f"{celex}{extension}")

    if basename and basename.upper() not in generic_endpoint_names:
        name = safe_filename(basename)
        if not Path(name).suffix:
            name = f"{name}{extension}"
        return name

    url_hash = hashlib.sha1(decoded.encode("utf-8")).hexdigest()[:10]
    return f"manual_download_{url_hash}{extension}"


records = []

for source_row_idx, row in manual_df.reset_index(drop=True).iterrows():
    source_row = source_row_idx + 1
    urls = split_urls(row.get("download_urls"))

    # Keep rows with missing URLs visible in the scientific audit trail.
    if not urls:
        urls = [""]

    for url_index, url in enumerate(urls, start=1):
        file_name = source_filename_from_url(url, source_row)

        records.append({
            "source_row": source_row,
            "url_index_in_row": url_index,
            "decision_date": row.get("decision_date"),
            "case_numbers_raw": row.get("case_numbers_raw"),
            "case_numbers_normalized": row.get("case_numbers_normalized"),
            "decision_label": row.get("decision_label"),
            "decision_title": row.get("decision_title"),
            "celex": extract_celex(url) if url else None,
            "source_url": url,
            "target_filename": file_name,
        })

download_df = pd.DataFrame(records)

# Prevent silent overwrites if different records resolve to the same filename.
duplicate_names = download_df["target_filename"].duplicated(keep=False)

if duplicate_names.any():
    for idx in download_df.index[duplicate_names]:
        row = download_df.loc[idx]
        path = Path(row["target_filename"])
        suffix = path.suffix
        stem = path.stem
        download_df.loc[idx, "target_filename"] = (
            f"{stem}__row{int(row['source_row'])}_url{int(row['url_index_in_row'])}{suffix}"
        )

download_df["target_path"] = download_df["target_filename"].map(
    lambda name: raw_dir / name
)

download_df["exists_before"] = download_df["target_path"].map(Path.exists)

print(f"Download records: {len(download_df):,}")
print(f"Already present:  {download_df['exists_before'].sum():,}")
print(f"To retrieve:      {(~download_df['exists_before'] | REDO_ALL).sum():,}")

download_df[
    [
        "source_row",
        "decision_title",
        "celex",
        "target_filename",
        "exists_before",
        "source_url",
    ]
].head(20)

Download records: 16
Already present:  0
To retrieve:      16


,source_row,decision_title,celex,target_filename,exists_before,source_url
0,1,National Panasonic France,31982D0465,31982D0465.html,False,https://eur-lex.europa.eu/legal-content/EN/TXT...
1,2,Anheuser-Busch Inc/Scottish and New Castle,32000D0146,32000D0146.html,False,https://eur-lex.europa.eu/legal-content/EN/TXT...
2,3,Zaventem,31995D0364,31995D0364.html,False,https://eur-lex.europa.eu/legal-content/EN/TXT...
3,4,Greek Insurance,31985D0276,31985D0276.html,False,https://eur-lex.europa.eu/legal-content/EN/TXT...
4,5,Canary Islands,31987D0359,31987D0359.html,False,https://eur-lex.europa.eu/legal-content/EN/TXT...
5,6,Provision in the Netherlands of express delive...,31990D0016,31990D0016.html,False,https://eur-lex.europa.eu/legal-content/EN/TXT...
6,7,Rodby,31994D0119,31994D0119.html,False,https://eur-lex.europa.eu/legal-content/EN/TXT...
7,8,GSM Italia,31995D0489,31995D0489.html,False,https://eur-lex.europa.eu/legal-content/EN/TXT...
8,9,Telecom Eireann,31997D0114,31997D0114.html,False,https://eur-lex.europa.eu/legal-content/EN/TXT...
9,10,GSM Spain,31997D0181,31997D0181.html,False,https://eur-lex.europa.eu/legal-content/EN/TXT...


## 4. Download and validate the source documents

EUR-Lex is handled more carefully than an ordinary direct file server.

For EUR-Lex URLs, the notebook:

1. opens the page with Playwright/Chromium;
2. tries the supplied URL and sensible HTML/PDF variants;
3. waits for the rendered legal document;
4. rejects common robot-verification, access-denied, and document-generation placeholders;
5. accepts a PDF only when it has a valid `%PDF` signature;
6. accepts HTML only when it contains plausible legal-document markers and sufficient text.

For non-EUR-Lex URLs, a normal `requests` download is used with the same basic validation checks.

Existing files are validated before reuse when `REDO_ALL = False`.

**Raw-data principle:** PDFs and direct files are saved as returned. For EUR-Lex HTML, the saved file is the rendered HTML DOM snapshot produced by the browser; body text may be used for validation but is not appended to or otherwise inserted into the saved HTML.

In [5]:
BAD_PAGE_MARKERS = [
    "javascript is disabled",
    "verify that you're not a robot",
    "verify that you are not a robot",
    "not a robot",
    "access denied",
    "forbidden",
    "checking your browser",
    "enable javascript",
    "captcha",
    "cloudflare",
    "request unsuccessful",
    "temporarily unavailable",
    "document is being generated",
    "please wait",
    "accepted for processing",
]

GOOD_HTML_MARKERS = [
    "celex",
    "commission decision",
    "commission regulation",
    "official journal",
    "article 85",
    "article 86",
    "article 101",
    "article 102",
    "european commission",
    "competition",
]


def is_eurlex_url(url: str) -> bool:
    return "eur-lex.europa.eu/legal-content/" in clean_value(url).lower()


def remove_query_param(url: str, param_name: str) -> str:
    url = clean_value(url)
    if not url:
        return url

    parsed = urlparse(url)
    query = [
        (key, value)
        for key, value in parse_qsl(parsed.query, keep_blank_values=True)
        if key.lower() != param_name.lower()
    ]

    return urlunparse(parsed._replace(query=urlencode(query, doseq=True)))


def eurlex_variant(url: str, wanted: str) -> str:
    assert wanted in {"HTML", "PDF"}

    url = clean_value(url)
    lower = url.lower()
    target = f"/TXT/{wanted}/"

    if f"/txt/{wanted.lower()}/" in lower:
        return url

    if "/txt/html/" in lower:
        return re.sub(r"/TXT/HTML/", target, url, flags=re.IGNORECASE)

    if "/txt/pdf/" in lower:
        return re.sub(r"/TXT/PDF/", target, url, flags=re.IGNORECASE)

    if "/txt/" in lower:
        return re.sub(r"/TXT/", target, url, count=1, flags=re.IGNORECASE)

    if "/html/" in lower:
        return re.sub(r"/HTML/", target, url, count=1, flags=re.IGNORECASE)

    if "/pdf/" in lower:
        return re.sub(r"/PDF/", target, url, count=1, flags=re.IGNORECASE)

    return url


def expand_download_candidates(url: str) -> list[str]:
    url = clean_value(url)
    if not url:
        return []

    if not is_eurlex_url(url):
        return [url]

    bases = [url]

    no_qid = remove_query_param(url, "qid")
    if no_qid and no_qid != url:
        bases.append(no_qid)

    candidates = []

    for base in bases:
        candidates.append(base)

        for fmt in ("HTML", "PDF"):
            variant = eurlex_variant(base, fmt)
            if variant:
                candidates.append(variant)

                variant_no_qid = remove_query_param(variant, "qid")
                if variant_no_qid and variant_no_qid != variant:
                    candidates.append(variant_no_qid)

    return list(dict.fromkeys(candidates))


def headers_for_url(url: str) -> dict:
    headers = dict(HEADERS)
    lower = clean_value(url).lower()

    if "/txt/pdf/" in lower:
        headers["Accept"] = (
            "application/pdf,application/octet-stream;q=0.9,*/*;q=0.8"
        )
    elif "/txt/html/" in lower:
        headers["Accept"] = (
            "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
        )

    return headers


def sniff_bytes_text(content: bytes, limit: int = 200_000) -> str:
    sample = (content or b"")[:limit]

    for encoding in ("utf-8", "cp1252", "latin-1"):
        try:
            return sample.decode(encoding, errors="ignore")
        except Exception:
            pass

    return ""


def classify_content(
    content: bytes,
    content_type: str = "",
    url: str = "",
    celex: str | None = None,
    extra_validation_text: str = "",
) -> tuple[bool, str]:
    if not content or len(content) < MIN_SUCCESS_BYTES:
        return False, "empty_or_too_small"

    lower_url = clean_value(url).lower()
    content_type_lower = clean_value(content_type).lower()
    head = content[:2048].lstrip()

    wants_pdf = (
        "/txt/pdf/" in lower_url
        or "application/pdf" in content_type_lower
        or lower_url.endswith(".pdf")
    )

    if wants_pdf:
        if head.startswith(b"%PDF"):
            return True, "valid_pdf_signature"

        text = sniff_bytes_text(content).lower()

        for marker in BAD_PAGE_MARKERS:
            if marker in text:
                return False, f"bad_page_marker:{marker}"

        return False, "pdf_candidate_without_pdf_signature"

    text = sniff_bytes_text(content)
    validation_text = f"{text}\n{extra_validation_text}".lower()

    for marker in BAD_PAGE_MARKERS:
        if marker in validation_text:
            return False, f"bad_page_marker:{marker}"

    if celex and clean_value(celex).lower() in validation_text:
        return True, "celex_marker_found"

    good_hits = [
        marker
        for marker in GOOD_HTML_MARKERS
        if marker in validation_text
    ]

    plain_text = re.sub(r"<[^>]+>", " ", text)
    plain_text = re.sub(r"\s+", " ", plain_text).strip()
    plain_length = max(len(plain_text), len(extra_validation_text.strip()))

    if good_hits and plain_length >= MIN_HTML_TEXT_CHARS:
        return True, f"good_html_markers:{','.join(good_hits[:3])}"

    looks_html = (
        "html" in content_type_lower
        or head[:50].lower().startswith((b"<!doctype html", b"<html"))
    )

    if looks_html:
        return False, f"html_without_legal_markers_text_chars={plain_length}"

    # For ordinary non-HTML direct files, accept non-empty content unless
    # a bad-page signal was found above.
    return True, "non_html_nonempty"


def candidate_target_path(rec: dict, candidate_url: str, content_type: str = "") -> Path:
    planned = Path(rec["target_filename"])
    lower = clean_value(candidate_url).lower()
    ctype = clean_value(content_type).lower()

    suffix = planned.suffix.lower() or ".bin"

    if "/txt/pdf/" in lower or "application/pdf" in ctype:
        suffix = ".pdf"
    elif "/txt/html/" in lower or "html" in ctype:
        suffix = ".html"
    elif "xml" in ctype:
        suffix = ".xml"
    elif "text/plain" in ctype:
        suffix = ".txt"

    return raw_dir / safe_filename(planned.with_suffix(suffix).name)


def possible_existing_paths(rec: dict) -> list[Path]:
    planned = Path(rec["target_filename"])
    stem = planned.stem

    suffixes = [
        planned.suffix.lower() or ".bin",
        ".html",
        ".htm",
        ".pdf",
        ".xml",
        ".txt",
        ".bin",
    ]

    paths = []
    for suffix in dict.fromkeys(suffixes):
        path = raw_dir / f"{stem}{suffix}"
        if path not in paths:
            paths.append(path)

    return paths


def find_existing_usable_file(rec: dict) -> tuple[Path | None, str]:
    for path in possible_existing_paths(rec):
        if not path.exists():
            continue

        content = path.read_bytes()

        if path.suffix.lower() == ".pdf":
            ctype = "application/pdf"
        elif path.suffix.lower() in {".html", ".htm"}:
            ctype = "text/html"
        elif path.suffix.lower() == ".xml":
            ctype = "application/xml"
        elif path.suffix.lower() == ".txt":
            ctype = "text/plain"
        else:
            ctype = ""

        usable, reason = classify_content(
            content,
            content_type=ctype,
            url=str(path),
            celex=rec.get("celex"),
        )

        if usable:
            return path, reason

    return None, "no_valid_existing_file"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def repo_relative(path: Path) -> str:
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except Exception:
        return path.as_posix()


def make_result(rec: dict) -> dict:
    return {
        **rec,
        "success": False,
        "status": None,
        "download_method": None,
        "resolved_url": "",
        "file_name": "",
        "raw_file_path": "",
        "file_format": "",
        "bytes": None,
        "sha256": "",
        "validation_reason": "",
        "error": "",
    }


def finish_success(
    result: dict,
    path: Path,
    status: str,
    method: str,
    resolved_url: str,
    validation_reason: str,
) -> dict:
    result.update({
        "success": True,
        "status": status,
        "download_method": method,
        "resolved_url": resolved_url,
        "file_name": path.name,
        "raw_file_path": repo_relative(path),
        "file_format": path.suffix.lower().lstrip("."),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "validation_reason": validation_reason,
        "error": "",
    })

    return result


def download_with_requests(session: requests.Session, rec: dict) -> dict:
    result = make_result(rec)

    if not clean_value(rec.get("source_url")):
        result.update({
            "status": "missing_url",
            "download_method": "none",
            "error": "No source URL supplied in input CSV.",
        })
        return result

    if not REDO_ALL:
        existing_path, reason = find_existing_usable_file(rec)
        if existing_path is not None:
            return finish_success(
                result,
                existing_path,
                status="existing",
                method="existing",
                resolved_url=rec["source_url"],
                validation_reason=reason,
            )

    errors = []

    for candidate_url in expand_download_candidates(rec["source_url"]):
        try:
            response = session.get(
                candidate_url,
                headers=headers_for_url(candidate_url),
                timeout=REQUEST_TIMEOUT_SECONDS,
                allow_redirects=True,
            )

            content = response.content or b""
            content_type = response.headers.get("Content-Type", "")

            usable, reason = classify_content(
                content,
                content_type=content_type,
                url=candidate_url,
                celex=rec.get("celex"),
            )

            if response.status_code == 200 and usable:
                target_path = candidate_target_path(
                    rec,
                    candidate_url,
                    content_type,
                )
                target_path.parent.mkdir(parents=True, exist_ok=True)
                target_path.write_bytes(content)

                return finish_success(
                    result,
                    target_path,
                    status="downloaded",
                    method="requests",
                    resolved_url=response.url,
                    validation_reason=reason,
                )

            errors.append(
                f"{candidate_url}: HTTP {response.status_code}; {reason}"
            )

        except Exception as exc:
            errors.append(f"{candidate_url}: {type(exc).__name__}: {exc}")

    result.update({
        "status": "failed",
        "download_method": "requests",
        "error": " | ".join(errors),
    })

    return result


async def accept_common_cookie_banners(page):
    labels = [
        "Accept all cookies",
        "Accept cookies",
        "Accept all",
        "Agree",
        "I agree",
        "Accepter",
        "Tout accepter",
        "Alle akzeptieren",
        "Akzeptieren",
    ]

    for label in labels:
        try:
            button = page.get_by_role(
                "button",
                name=re.compile(label, re.I),
            )

            if await button.count():
                await button.first.click(timeout=1500)
                await page.wait_for_timeout(500)
                return

        except Exception:
            pass


async def fetch_eurlex_candidate(page, candidate_url: str, rec: dict) -> dict:
    info = {
        "status_code": None,
        "content_type": "",
        "content": b"",
        "resolved_url": candidate_url,
        "validation_reason": "",
        "error": "",
    }

    lower = clean_value(candidate_url).lower()
    wants_pdf = "/txt/pdf/" in lower or lower.endswith(".pdf")

    if wants_pdf:
        try:
            response = await page.context.request.get(
                candidate_url,
                headers=headers_for_url(candidate_url),
                timeout=PLAYWRIGHT_TIMEOUT_MS,
            )

            content = await response.body()
            content_type = response.headers.get("content-type", "")

            usable, reason = classify_content(
                content,
                content_type=content_type,
                url=candidate_url,
                celex=rec.get("celex"),
            )

            info.update({
                "status_code": response.status,
                "content_type": content_type,
                "content": content,
                "resolved_url": response.url,
                "validation_reason": reason,
            })

            return info

        except Exception as exc:
            info.update({
                "validation_reason": "pdf_browser_request_exception",
                "error": f"{type(exc).__name__}: {exc}",
            })
            return info

    try:
        response = await page.goto(
            candidate_url,
            wait_until="domcontentloaded",
            timeout=PLAYWRIGHT_TIMEOUT_MS,
        )

        if response is not None:
            info["status_code"] = response.status
            info["content_type"] = response.headers.get(
                "content-type",
                "text/html",
            )

        info["resolved_url"] = page.url

        await accept_common_cookie_banners(page)

        for attempt in range(PLAYWRIGHT_MAX_RENDER_RETRIES):
            await page.wait_for_timeout(
                int(PLAYWRIGHT_RENDER_WAIT_SECONDS * 1000)
            )

            try:
                await page.wait_for_load_state(
                    "networkidle",
                    timeout=15_000,
                )
            except Exception:
                pass

            if PLAYWRIGHT_EXTRA_SETTLE_SECONDS:
                await page.wait_for_timeout(
                    int(PLAYWRIGHT_EXTRA_SETTLE_SECONDS * 1000)
                )

            html = await page.content()

            try:
                body_text = await page.locator("body").inner_text(
                    timeout=5_000
                )
            except Exception:
                body_text = ""

            # Save the rendered HTML snapshot itself. Body text is used only
            # for validation and is not inserted into the raw file.
            content = html.encode("utf-8", errors="ignore")

            usable, reason = classify_content(
                content,
                content_type="text/html",
                url=page.url,
                celex=rec.get("celex"),
                extra_validation_text=body_text,
            )

            info.update({
                "content": content,
                "resolved_url": page.url,
                "validation_reason": reason,
            })

            if usable:
                return info

            if attempt < PLAYWRIGHT_MAX_RENDER_RETRIES - 1:
                await page.wait_for_timeout(
                    int(PLAYWRIGHT_RETRY_SLEEP_SECONDS * 1000)
                )

                try:
                    await page.reload(
                        wait_until="domcontentloaded",
                        timeout=PLAYWRIGHT_TIMEOUT_MS,
                    )
                except Exception:
                    pass

        return info

    except Exception as exc:
        info.update({
            "validation_reason": "html_browser_navigation_exception",
            "error": f"{type(exc).__name__}: {exc}",
        })

        try:
            info["resolved_url"] = page.url
        except Exception:
            pass

        return info


async def download_with_playwright(page, rec: dict) -> dict:
    result = make_result(rec)

    if not clean_value(rec.get("source_url")):
        result.update({
            "status": "missing_url",
            "download_method": "none",
            "error": "No source URL supplied in input CSV.",
        })
        return result

    if not REDO_ALL:
        existing_path, reason = find_existing_usable_file(rec)
        if existing_path is not None:
            return finish_success(
                result,
                existing_path,
                status="existing",
                method="existing",
                resolved_url=rec["source_url"],
                validation_reason=reason,
            )

    errors = []

    for candidate_url in expand_download_candidates(rec["source_url"]):
        fetched = await fetch_eurlex_candidate(
            page,
            candidate_url,
            rec,
        )

        content = fetched.get("content") or b""
        content_type = fetched.get("content_type") or ""
        resolved_url = fetched.get("resolved_url") or candidate_url

        usable, reason = classify_content(
            content,
            content_type=content_type,
            url=resolved_url,
            celex=rec.get("celex"),
        )

        # An initial EUR-Lex navigation status can be 202. Content validation
        # therefore determines success rather than the status code alone.
        if usable:
            target_path = candidate_target_path(
                rec,
                candidate_url,
                content_type,
            )
            target_path.parent.mkdir(parents=True, exist_ok=True)
            target_path.write_bytes(content)

            return finish_success(
                result,
                target_path,
                status="downloaded",
                method="playwright",
                resolved_url=resolved_url,
                validation_reason=fetched.get("validation_reason") or reason,
            )

        detail = fetched.get("error") or ""
        status_code = fetched.get("status_code")

        errors.append(
            f"{candidate_url}: status={status_code}; "
            f"{fetched.get('validation_reason') or reason}"
            + (f"; {detail}" if detail else "")
        )

    result.update({
        "status": "failed",
        "download_method": "playwright",
        "error": " | ".join(errors),
    })

    return result


async def run_downloads(download_df: pd.DataFrame) -> pd.DataFrame:
    needs_eurlex = download_df["source_url"].map(is_eurlex_url).any()

    if needs_eurlex and async_playwright is None:
        raise RuntimeError(
            "This input contains EUR-Lex URLs but Playwright is not installed. "
            "Run: pip install playwright && playwright install chromium"
        )

    results = []

    session = requests.Session()
    session.headers.update(HEADERS)

    if needs_eurlex:
        async with async_playwright() as playwright:
            browser = await playwright.chromium.launch(headless=HEADLESS)

            context = await browser.new_context(
                user_agent=USER_AGENT,
                locale="en-US",
                viewport={"width": 1400, "height": 1000},
                java_script_enabled=True,
                accept_downloads=True,
                extra_http_headers={
                    "Accept-Language": "en-US,en;q=0.9,de;q=0.8,fr;q=0.7"
                },
            )

            page = await context.new_page()
            page.set_default_timeout(PLAYWRIGHT_TIMEOUT_MS)

            for rec in tqdm(
                download_df.to_dict("records"),
                desc="Downloading com_manual",
            ):
                if is_eurlex_url(rec.get("source_url")):
                    result = await download_with_playwright(page, rec)
                else:
                    result = download_with_requests(session, rec)

                results.append(result)

                delay = (
                    random.uniform(*SLEEP_SECONDS)
                    if isinstance(SLEEP_SECONDS, tuple)
                    else float(SLEEP_SECONDS or 0)
                )

                if delay:
                    await asyncio.sleep(delay)

            await context.close()
            await browser.close()

    else:
        for rec in tqdm(
            download_df.to_dict("records"),
            desc="Downloading com_manual",
        ):
            result = download_with_requests(session, rec)
            results.append(result)

            delay = (
                random.uniform(*SLEEP_SECONDS)
                if isinstance(SLEEP_SECONDS, tuple)
                else float(SLEEP_SECONDS or 0)
            )

            if delay:
                await asyncio.sleep(delay)

    return pd.DataFrame(results)

## 5. Run the downloads

This cell performs the actual acquisition.

Re-running the notebook is safe by default: when `REDO_ALL = False`, an existing file is validated and reused rather than downloaded again.

In [6]:
download_results = await run_downloads(download_df)

print(
    f"Successful / available: "
    f"{int(download_results['success'].sum()):,} / {len(download_results):,}"
)

download_results[
    [
        "source_row",
        "decision_title",
        "celex",
        "success",
        "status",
        "file_name",
        "raw_file_path",
    ]
].head(20)

Successful / available: 16 / 16


,source_row,decision_title,celex,success,status,file_name,raw_file_path
0,1,National Panasonic France,31982D0465,True,downloaded,31982D0465.html,data/raw/com_manual/files/31982D0465.html
1,2,Anheuser-Busch Inc/Scottish and New Castle,32000D0146,True,downloaded,32000D0146.html,data/raw/com_manual/files/32000D0146.html
2,3,Zaventem,31995D0364,True,downloaded,31995D0364.html,data/raw/com_manual/files/31995D0364.html
3,4,Greek Insurance,31985D0276,True,downloaded,31985D0276.html,data/raw/com_manual/files/31985D0276.html
4,5,Canary Islands,31987D0359,True,downloaded,31987D0359.html,data/raw/com_manual/files/31987D0359.html
5,6,Provision in the Netherlands of express delive...,31990D0016,True,downloaded,31990D0016.html,data/raw/com_manual/files/31990D0016.html
6,7,Rodby,31994D0119,True,downloaded,31994D0119.html,data/raw/com_manual/files/31994D0119.html
7,8,GSM Italia,31995D0489,True,downloaded,31995D0489.html,data/raw/com_manual/files/31995D0489.html
8,9,Telecom Eireann,31997D0114,True,downloaded,31997D0114.html,data/raw/com_manual/files/31997D0114.html
9,10,GSM Spain,31997D0181,True,downloaded,31997D0181.html,data/raw/com_manual/files/31997D0181.html


## 6. Save the compact download manifest

The manifest is intentionally small enough to inspect by eye, while retaining the information needed for scientific provenance and the next processing step.

It contains the original decision metadata, source URL, resolved URL, local raw-file path, success/status information, download method, file size, and SHA-256 checksum.

The key handoff fields for the next notebook are:

- `success`
- `file_name`
- `raw_file_path`

Only successfully acquired files should be passed to text extraction/OCR.

In [7]:
manifest_columns = [
    "source_row",
    "url_index_in_row",
    "decision_date",
    "case_numbers_raw",
    "case_numbers_normalized",
    "decision_label",
    "decision_title",
    "celex",
    "source_url",
    "resolved_url",
    "file_name",
    "raw_file_path",
    "success",
    "status",
    "download_method",
    "file_format",
    "bytes",
    "sha256",
    "error",
]

manifest = download_results[manifest_columns].copy()

manifest.to_csv(manifest_path, index=False)

summary = (
    manifest["status"]
    .fillna("unknown")
    .value_counts()
    .rename_axis("status")
    .reset_index(name="count")
)

print(f"Saved manifest: {manifest_path}")
print(f"Manifest rows : {len(manifest):,}")
print(f"Successes     : {int(manifest['success'].sum()):,}")
print(f"Failures      : {int((~manifest['success']).sum()):,}")

display(summary)

failures = manifest.loc[
    ~manifest["success"],
    [
        "source_row",
        "decision_title",
        "source_url",
        "status",
        "error",
    ],
]

if len(failures):
    print("\nRecords requiring attention:")
    display(failures)
else:
    print("\nAll download records are available.")

Saved manifest: /home/edik/projects/ec_cjeu/output/com_manual/com_manual_download_manifest.csv
Manifest rows : 16
Successes     : 16
Failures      : 0


,status,count
0,downloaded,16



All download records are available.


## 7. Handoff to text processing

At the end of this notebook there are two scientific-data products:

### Raw source documents

`data/raw/com_manual/files/`

These files should be treated as the acquisition layer and left unchanged by later processing.

### Download manifest

`output/com_manual/com_manual_download_manifest.csv`

This is the registry connecting the manually curated Commission list to the locally preserved source files.

The subsequent `process_com_manual` notebook should read this manifest, keep rows where `success == True`, use `raw_file_path` as its input, and produce canonical cleaned `.txt` files plus a separate clean-file manifest.

No OCR or text cleaning is performed here.